# IMAS Colormap Playground

This notebook is a lightweight place to compare colormaps on one selected IMAS time slice.

It is written around one locally saved single-case IMAS file, so you can quickly play with field names, norms, and colormaps without dealing with a full discharge first.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm, Normalize

from hdg_postprocess.imas_export.reader import (
    equilibrium_slice,
    field_style_presets,
    load_ids,
    plasma_slice,
    plot_field_2d,
    symmetric_limits,
)


## Choose one saved slice

Set the path to one locally saved single-case IMAS file here. For a steady export, keep `time_index = 0`.


In [ ]:
db_path = "path/to/imas_single_case.nc"
occurrence = 0
time_index = 0
separatrix_level = None  # Set if you know the correct psi contour level for this export.

summary, equilibrium, plasma = load_ids(db_path, occurrence=occurrence)
eq = equilibrium_slice(equilibrium, plasma, time_index=time_index)
pl = plasma_slice(plasma, time_index=time_index)
styles = field_style_presets()


## Small plotting helpers


In [ ]:
def positive_log_norm(field):
    values = np.asarray(field, dtype=float)
    finite = values[np.isfinite(values) & (values > 0)]
    if finite.size == 0:
        return None
    return LogNorm(vmin=float(finite.min()), vmax=float(finite.max()))


def default_norm(field_name, field):
    style = styles[field_name]
    if style["scale"] == "log":
        return positive_log_norm(field)
    if style.get("symmetric"):
        vmin, vmax = symmetric_limits(field)
        return Normalize(vmin=vmin, vmax=vmax)
    values = np.asarray(field, dtype=float)
    finite = values[np.isfinite(values)]
    if finite.size == 0:
        return None
    return Normalize(vmin=float(finite.min()), vmax=float(finite.max()))


def separatrix_overlay(level):
    if level is None:
        return None
    return {"psi": eq["psi"], "level": float(level)}


def compare_cmaps(field, *, field_name, cmaps, norm=None, title_prefix=""):
    sep = separatrix_overlay(separatrix_level)
    fig, axes = plt.subplots(1, len(cmaps), figsize=(5 * len(cmaps), 4.8), constrained_layout=True)
    if len(cmaps) == 1:
        axes = [axes]
    for ax, cmap in zip(axes, cmaps):
        mesh, label = plot_field_2d(
            ax,
            r=eq["r"],
            z=eq["z"],
            field=field,
            title=f"{title_prefix}{cmap}",
            label=field_name,
            cmap=cmap,
            norm=norm,
            separatrix=sep,
        )
        fig.colorbar(mesh, ax=ax, label=label)
    plt.show()


## Log-scale plasma fields

`inferno`, `magma`, and `cividis` are a good first comparison set for `n_e`, `n_n`, `T_e`, and `T_i`.


In [ ]:
field_name = "te"  # Try: ne, nn, te, ti
field = pl[field_name]
compare_cmaps(
    field,
    field_name=field_name,
    cmaps=["inferno", "magma", "cividis"],
    norm=default_norm(field_name, field),
    title_prefix=f"{field_name} with ",
)


## Signed fields symmetric around zero

`bwr` is a good starting point for `u_par`, `B_R`, `B_Z`, and similar fields where the sign structure matters.


In [ ]:
field_name = "bz"  # Try: u_par, br, bz, jphi
field = eq[field_name] if field_name in eq else pl[field_name]
compare_cmaps(
    field,
    field_name=field_name,
    cmaps=["bwr", "RdBu_r", "coolwarm"],
    norm=default_norm(field_name, field),
    title_prefix=f"{field_name} with ",
)


## Poloidal flux

For `psi`, a contour view is often more informative than a plain color map, especially when you want to highlight the separatrix.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

mesh, label = plot_field_2d(
    axes[0],
    r=eq["r"],
    z=eq["z"],
    field=eq["psi"],
    title="psi with cividis",
    label="psi",
    cmap="cividis",
    norm=default_norm("psi", eq["psi"]),
    separatrix=separatrix_overlay(separatrix_level),
)
fig.colorbar(mesh, ax=axes[0], label=label)

contours = axes[1].contour(eq["r"], eq["z"], eq["psi"], levels=20, cmap="cividis")
if separatrix_level is not None:
    axes[1].contour(eq["r"], eq["z"], eq["psi"], levels=[float(separatrix_level)], colors="black", linewidths=1.2)
fig.colorbar(contours, ax=axes[1], label="psi")
axes[1].set_aspect("equal")
axes[1].set_xlabel("R [m]")
axes[1].set_ylabel("Z [m]")
axes[1].set_title("psi contours")

plt.show()


## Notes

- Outside the HDG mesh, exported values are `NaN`.
- This notebook is intentionally built around one single saved case, so it is quick to use locally.
- If you later switch to a full discharge, the same reader helpers will still resolve repeated-grid IMAS `path` references for you.
- `separatrix_level` is not inferred automatically from the export; set it manually when you know the correct `psi` contour value for your case.
